#### Notebook to fetch data from Storage bucket, process and upload to dashboard. 

In [1]:
from pathlib import Path
from spending_tracker.data_processing.statements import parse_and_clean_batch
from spending_tracker.database import transactions as transactions_db
from spending_tracker.database import supabase_client
from spending_tracker.data_ingestion import csv_parsing

In [2]:
data_path = Path().resolve().parent / "data"
month = "September-26"
bucket_name = "spending-tracker"

In [3]:
bucket = supabase_client.load_bucket(bucket_name)

#### Clean Transactions and generate Metadata

In [5]:
data_fetched = transactions_db.fetch_transactions_from_bucket(month=month,
                                               bucket=bucket,
                                               data_home=data_path)

for key in data_fetched:
    print(key)

data/transactions/raw/Taj/Amex/52006/September-26/statement.csv
data/transactions/raw/Taj/Chase/33053200/September-26/statement.csv


In [4]:
parse_and_clean_batch(data_home=data_path,
                      specific_month=month)

Processing Taj/Chase/33053200/September-26


#### Upload to Database

In [ ]:
db_client = supabase_client.load_supabase_client()

In [ ]:
processed_transactions_data_home = data_path / "transactions" / "processed"
transaction_metadata_home = data_path / "transactions_metadata"

In [ ]:
transactions_df = csv_parsing.load_csv_in_batch(processed_transactions_data_home,
                                                specific_month=month,
                                                dtype_kwargs={
                                                    "account_id": str,
                                                })

transactions_metadata_df = csv_parsing.load_csv_in_batch(transaction_metadata_home,
                                                         specific_month=month,
                                                         dtype_kwargs={
                                                             "account_id": str,
                                                         })

In [ ]:
transactions_df.head(10)

In [ ]:
transactions_metadata_df.head(10)

In [ ]:
transactions_df.shape

In [ ]:
transactions_metadata_df.shape

In [ ]:
transactions_db.upload_transactions_to_db(transactions_df, 
                                       db_client, 
                                       table_name="transactions")
transactions_db.upload_transactions_to_db(transactions_metadata_df, 
                                       db_client, 
                                       table_name="transactions_metadata")